# 12_Feature_Relationships

### What are Feature Relationships?

Feature relationships describe how input variables in a dataset relate to each other and to the target variable. Understanding these relationships helps identify redundant features, diagnose model instability, and select the most useful predictors before training a machine learning model.

Example

In a housing price dataset, square footage and number of rooms are strongly related to each other, while both are also related to the final sale price — understanding these relationships is key before building a prediction model.

## 01. Correlation Matrix

**Definition**: A Correlation Matrix is a table showing the correlation coefficients between multiple numeric variables, indicating the strength and direction of their linear relationships.

**Formula**: r = Cov(X,Y) / (σX × σY), computed pairwise for every combination of variables in the dataset.

**Where it is used**: Used during exploratory data analysis (EDA) to understand relationships between features and to detect potential multicollinearity before modeling.

**Real-World Example**: A marketing team uses a correlation matrix to see how strongly ad spend, website visits, and email campaigns correlate with sales revenue.

**AI/ML Example**: Correlation matrices are used before model training to identify highly correlated features that may be redundant and could be removed or combined.

In [9]:
import pandas as pd
df = pd.DataFrame({
    'sqft': [1000, 1200, 1500, 1800, 2000],
    'rooms': [2, 3, 3, 4, 5],
    'price': [150000, 180000, 225000, 270000, 300000]
})
print(df.corr())

          sqft    rooms    price
sqft   1.00000  0.95723  1.00000
rooms  0.95723  1.00000  0.95723
price  1.00000  0.95723  1.00000


**Interpretation**: The correlation matrix shows that **sqft, rooms, and price have a strong positive relationship with each other**.


## 02. Multicollinearity

**Definition**: Multicollinearity occurs when two or more independent (predictor) variables in a regression model are highly correlated with each other, making it difficult to isolate the individual effect of each variable on the target.

**Formula**: No single formula; commonly assessed using a correlation matrix or the Variance Inflation Factor (VIF).

**Where it is used**: Checked during regression modeling to ensure the stability and interpretability of estimated coefficients.

**Real-World Example**: In a house price model, square footage and number of rooms are highly correlated, so it becomes hard to tell which one is truly driving the price increase.

**AI/ML Example**: Multicollinearity inflates the variance of regression coefficients, making feature importance and coefficient interpretation unreliable in linear models.

In [10]:
import pandas as pd
print(df[['sqft', 'rooms']].corr())

          sqft    rooms
sqft   1.00000  0.95723
rooms  0.95723  1.00000


**Interpretation**: High correlation between sqft and rooms indicates multicollinearity

## 03. Variance Inflation Factor (VIF)

**Definition**: VIF quantifies how much the variance of a regression coefficient is inflated due to multicollinearity with the other predictor variables in the model.

**Formula**: VIF_i = 1 / (1 − R_i²), where R_i² is the R-squared value obtained by regressing feature i against all other features.

**Where it is used**: Used in regression diagnostics to detect and quantify the severity of multicollinearity among predictors.

**Real-World Example**: A salary prediction model checks the VIF of "years of experience" and "age," since older employees also tend to have more experience.

**AI/ML Example**: VIF is used in automated feature engineering pipelines to flag and drop features with VIF above 10 before fitting a linear regression model.

In [11]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df[['sqft', 'rooms']]
vif = pd.DataFrame()
vif['Feature'] = X.columns
vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif)

  Feature    VIF
0    sqft  119.3
1   rooms  119.3


**Interpretation**: VIF > 5 indicates possible multicollinearity, while VIF > 10 indicates high multicollinearity.

## 04. Feature Importance

**Definition**: Feature Importance is a technique that scores each input feature based on how much it contributes to a model's predictions, helping identify the most influential variables.

**Formula**: No single formula; common methods include tree-based importance (mean decrease in impurity), permutation importance, and coefficient magnitude in linear models.

**Where it is used**: Used after model training to interpret predictions, explain model behavior to stakeholders, and guide feature selection.

**Real-World Example**: A bank uses feature importance to explain why a loan approval model rejected an application, showing that credit history was the most influential factor.

**AI/ML Example**: Feature importance scores are used to select the top contributing features, reducing dimensionality and improving model training speed without losing much accuracy.

In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
np.random.seed(42)
credit_history = np.random.randint(0,2,200)
income = np.random.normal(50000,15000,200)
age = np.random.randint(21,60,200)
loan_default = (credit_history==0).astype(int)
X = pd.DataFrame({'credit_history':credit_history,'income':income,'age':age})
y = loan_default
model = RandomForestClassifier(random_state=42)
model.fit(X,y)
print(model.feature_importances_)

[0.89143252 0.06392412 0.04464337]


**Interpretation**: Credit history shows the highest importance score, confirming it is the most influential feature in predicting loan default, while income and age contribute comparatively less.

## 05. Feature Selection Basics

**Definition**: Feature Selection is the process of choosing a subset of the most relevant features from the original dataset to use in model building, improving performance and reducing overfitting.

**Formula**: No single formula; approaches include filter methods (correlation, chi-square), wrapper methods (Recursive Feature Elimination), and embedded methods (Lasso regularization).

**Where it is used**: Used in preprocessing pipelines to reduce dimensionality, remove noisy or redundant features, and speed up model training.

**Real-World Example**: A telecom company selects the most relevant features — call duration, number of complaints, and tenure — to predict customer churn instead of using all available columns.

**AI/ML Example**: Feature selection techniques like Recursive Feature Elimination (RFE) are used to automatically identify and keep only the most predictive features before training a model.

In [13]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
np.random.seed(42)
call_duration = np.random.normal(300,50,200)
complaints = np.random.poisson(2,200)
tenure = np.random.normal(24,10,200)
random_noise = np.random.normal(0,1,200)
churn = (complaints>2).astype(int)
X = pd.DataFrame({'call_duration':call_duration,'complaints':complaints,'tenure':tenure,'random_noise':random_noise})
y = churn
selector = SelectKBest(score_func=f_classif,k=2)
selector.fit(X,y)
print(X.columns[selector.get_support()])

Index(['complaints', 'tenure'], dtype='str')


**Interpretation**: The feature selection method correctly identifies "complaints" as one of the top predictive features (since churn was defined based on it), while filtering out the random noise column that carries no real predictive value.

## Overall Advantages and Limitations of Feature Relationship Analysis

### Advantages
1. Helps identify redundant or highly correlated features early, before they cause model instability.
2. Improves model interpretability by revealing which features actually drive predictions.
3. Reduces overfitting and training time by removing irrelevant or noisy features.
4. Diagnostic tools like VIF give a quantifiable measure of multicollinearity severity, not just a rough estimate.
5. Feature importance and selection techniques make it easier to explain model decisions to non-technical stakeholders.

### Limitations
1. Correlation only captures linear relationships and can miss important non-linear dependencies between features.
2. High VIF indicates multicollinearity but doesn't say which of the correlated features should be removed — domain knowledge is still needed.
3. Tree-based feature importance can be biased toward features with more unique values or higher cardinality.
4. Removing correlated or "unimportant" features without care can sometimes discard information useful in combination with other features.
5. Feature selection results can vary between methods (filter vs wrapper vs embedded), so results should be validated with model performance.

## Interview Questions & Answers

**1. What does a correlation matrix tell you, and what are its limitations?**
A correlation matrix shows the pairwise linear relationship strength between numeric variables, but its main limitation is that it only captures linear relationships and can completely miss strong non-linear associations between features.

**2. What is multicollinearity, and why is it a problem in regression models?**
Multicollinearity occurs when predictor variables are highly correlated with each other, and it's a problem because it inflates the variance of coefficient estimates, making them unstable and difficult to interpret individually.

**3. How is VIF calculated, and what threshold is typically used to flag a problematic feature?**
VIF is calculated as 1/(1−R²), where R² comes from regressing one feature against all others, and a VIF above 5 or 10 is commonly used as a threshold to flag significant multicollinearity.

**4. What is the difference between feature importance and feature selection?**
Feature importance scores how much each feature contributes to a trained model's predictions, while feature selection is the broader process of choosing which subset of features to actually use before or during model training.

**5. What are the three main types of feature selection methods?**
The three main types are filter methods (using statistical measures like correlation or chi-square independent of any model), wrapper methods (like Recursive Feature Elimination, which use model performance to select features), and embedded methods (like Lasso, which perform selection during model training).

**6. Why might two different feature importance methods rank the same features differently?**
Different methods measure importance differently — for example, tree-based impurity importance can be biased toward high-cardinality features, while permutation importance measures the actual drop in model performance, so they can disagree on ranking.

**7. If two features have a VIF above 10, what are your options to address it?**
Options include removing one of the two correlated features, combining them into a single feature (like a ratio or sum), applying dimensionality reduction (like PCA), or using regularized models (like Ridge regression) that are more robust to multicollinearity.

## Self-Reflection

Working through this notebook helped me understand that features rarely exist in isolation — they interact with each other in ways that can silently distort a model if left unchecked. The key insight I take away is that correlation and VIF serve as early diagnostic tools to catch multicollinearity before it damages model interpretability, while feature importance and selection help focus the model on what actually matters after or during training. I also now understand that no single method (correlation, VIF, or importance score) should be trusted in isolation — they each reveal a different angle of the same underlying relationships. This connects directly to real ML workflows, where careful feature relationship analysis during preprocessing often has a bigger impact on model quality than switching to a more complex algorithm.